# fastmux

> Drive and inspect tmux from Python: live session, window, and pane handles with CLI-style reprs

fastmux is a pythonic binding to [tmux](https://github.com/tmux/tmux): `tmux()` shows every session, window, and pane as a tree, and `tmux(target)` (using standard tmux target syntax such as `mysess:1.2`, `%5`, or `@3`) returns a live `Session`, `Window`, or `Pane` handle. A `Pane`'s repr is its current screen; index or slice it to read the transcript; search any scope rg-style; and drive panes with `send`, `send_keys`, and friends. It is the foundation that [bgtmux](https://github.com/AnswerDotAI/bgtmux) builds its managed background-session layer on.

## Install

```sh
pip install fastmux
```

You will also need `tmux` itself installed and on your `PATH`.

## Getting started

> **Status**: fastmux is young. It was extracted from [bgtmux](https://github.com/AnswerDotAI/bgtmux), which will become a thin managed-jobs layer on top of it; until that lands, bgtmux still carries its own copy of this functionality.


In [ ]:
import sys
from fastmux import *

Start a throwaway session running anything you like -- it's created detached, so your terminal is untouched:

In [ ]:
s = new_session([sys.executable,'-u','-c','import time\nfor i in range(30): print(f"line {i}")\ntime.sleep(600)'],
                width=60, height=8)
p = s.pane
p.poll(wait_ms=2000)  # wait for output to arrive
p

```python
line 23
line 24
line 25
line 26
line 27
line 28
line 29
```

A `Pane`'s repr is its live screen. The whole transcript (scrollback included) works like a list of lines:

In [ ]:
len(p), p[0], p[-1], p[5:7].lines

(30, 'line 0', 'line 29', ('line 5', 'line 6'))

Split panes with directional verbs (`rsplit`, `bsplit`, `lsplit`, `asplit` -- no tmux `-h`/`-v` confusion), again without stealing focus:

In [ ]:
p.bsplit(size=3, cmd='htop')
s.windows

1: ipyshell* (2 panes) @123
  1.1: [60x4] %153 python3 (active)
  1.2: [60x3] %154 tmux

Search any scope -- one pane, a window, a session, or every terminal you have -- and get rg-style hits whose `target` pastes straight back into `tmux()`:

In [ ]:
hits = s.search('line 2')
hits

110:1.1:2: line 2
110:1.1:20: line 20
110:1.1:21: line 21
110:1.1:22: line 22
110:1.1:23: line 23
110:1.1:24: line 24
110:1.1:25: line 25
110:1.1:26: line 26
110:1.1:27: line 27
110:1.1:28: line 28
110:1.1:29: line 29

In [ ]:
tmux(hits[0].target)[-3:]

```python
line 27
line 28
line 29
── 110:1.1 %153 · lines 27-30 of 30
```

And `tmux()` alone shows everything as a tree -- sessions, windows, panes:

In [ ]:
tmux()

110: 1 windows
  1: ipyshell* (2 panes) @123
    1.1: [60x4] %153 python3 (active)
    1.2: [60x3] %154 htop (dead)
default: 1 windows (attached)
  1: ipythonng* (4 panes) @0
    1.1: [122x29] %0 python3
    1.2: [122x26] %2 python3 · ⠂ Integrate Codex subscription with ipyshell AI
    1.3: [115x29] %1 python3 · ✳ Add auto-save feature with debouncing to SolveIt
    1.4: [238x56] %10 python3 (active) · ✳ Debug CTRL+Z suspension issue in ipythonng

In [ ]:
s.kill()

Panes can also be driven directly: `p.send('ls\n')` pastes text and polls for the response, `p.send_keys('C-c')` speaks tmux key names, and `p.wait()` returns an exit status. See the [full docs](https://AnswerDotAI.github.io/fastmux/core.html) for the complete API.